<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
《<a href="https://mng.bz/lZ5B">从零构建推理模型</a>》一书的配套代码，作者：<a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库： <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 第 5 章：通过自我改进实现推理时扩展

本 notebook 使用的软件包：

In [ ]:
from importlib.metadata import version

used_libraries = [
    "matplotlib",
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # 由 reasoning_from_scratch 使用
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch05/CH05_F01_raschka.webp" width=600>

&nbsp;
## 5.1 对模型回复进行评分并迭代改进

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch05/CH05_F02_raschka.webp" width=600>

- 本节没有代码

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch05/CH05_F03_raschka.webp" width=600>

&nbsp;
## 5.2 加载预训练模型

- 模型加载代码与第 2–4 章类似

- 注意，下面将设备硬编码为 `"cpu"`，以尽可能保证不同计算机和操作系统上的结果相近
- 建议先按原样运行本章代码，再删除下面的 `device = torch.device("cpu")`
- 即使将设备硬编码为 `"cpu"`，由于本章涉及大量极小数值的乘法，结果在你的计算机上仍可能有所不同（尤其是第 5.4 节及之后）


In [ ]:
import torch

from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (
     load_model_and_tokenizer
)

device = get_device()
device = torch.device("cpu")  # 可选：删除此设置以允许在 GPU 上运行

model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False
)

- 下面用上一章使用过的 MATH-500 数据集提示词测试模型：

In [ ]:
from reasoning_from_scratch.ch03 import render_prompt
from reasoning_from_scratch.ch04 import (
    generate_text_stream_concat_flex,
    generate_text_top_p_stream_cache
)

raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)
prompt_cot = prompt + "\n\nExplain step by step."

torch.manual_seed(0)
response_1 = generate_text_stream_concat_flex(
    model, tokenizer, prompt_cot, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_top_p_stream_cache,
    temperature=0.9,
    top_p=0.9
)

In [ ]:
torch.manual_seed(3)
response_2 = generate_text_stream_concat_flex(
    model, tokenizer, prompt_cot, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_top_p_stream_cache,
    temperature=0.9,
    top_p=0.9, 
)

In [ ]:
print("Response 1 characters:", len(response_1))
print("Response 1 tokens:", len(tokenizer.encode(response_1)))
print("\nResponse 2 characters:", len(response_2))
print("Response 2 tokens:", len(tokenizer.encode(response_2)))

- 顺便说明，正确答案是 83

&nbsp;
## 5.3 使用基于规则的分数评估 LLM 回复

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch05/CH05_F04_raschka.webp" width=600>

- 可以用作自洽性方法中的平局决胜规则
- 也可以对自洽性方法生成的所有回复进行评分和排序（这称为 Best-of-N 推理时扩展）


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch05/CH05_F05_raschka.webp" width=600>

In [ ]:
from reasoning_from_scratch.ch03 import extract_final_candidate
import math

def heuristic_score(
    answer,
    prompt=None,  # 会被忽略的占位符
    brevity_bonus=500.0,
    boxed_bonus=2.0,
    extract_bonus=1.0,
    fulltext_bonus=0.0,
):
    score = 0.0

    # 奖励包含最终方框值的答案
    cand = extract_final_candidate(answer, fallback="none")
    if cand:
        score += boxed_bonus

    # 如果答案没有方框值，则给予较弱奖励
    else:
        cand = extract_final_candidate(answer, fallback="number_only")
        if cand:
            score += extract_bonus
        else:
            cand = extract_final_candidate(
                answer, fallback="number_then_full"
            )
            if cand:
                score += fulltext_bonus

    # 添加随文本长度衰减的简洁奖励
    score += 1.5 * math.exp(-len(answer) / brevity_bonus)
    return score

- 为了简单起见，代码统计字符数而非词元数，因此这里不必传入分词器；如果需要，也很容易改为统计词元。

In [ ]:
import matplotlib.pyplot as plt

def plot_brevity_curve(brevity_bonus, max_len=2048):
    lengths = torch.arange(1, max_len)
    scores = 1.5 * torch.exp(-lengths / brevity_bonus)

    plt.figure(figsize=(4, 3))
    plt.plot(lengths, scores)
    plt.xlabel("Text length (number of characters)")
    plt.ylabel("Score contribution")
    plt.tight_layout()
    #plt.savefig("brevity_curve.pdf")
    plt.show()

plot_brevity_curve(500)

In [ ]:
print(round(heuristic_score(response_1), 3))

In [ ]:
print(round(heuristic_score(response_2), 3))

&nbsp;
## 5.4 理解词元概率分数

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch05/CH05_F07_raschka.webp" width=600>

- 词元概率分数也称为词元似然
- 实现一个为生成文本评分的函数
- 后续的自我改进方法会使用它；与 `heuristic_score` 类似，它也可以在自洽性方法中用作平局决胜规则
- 从技术上讲，为提高效率，可以在第 4.5 节的 `generate_text_top_p_stream_cache` 函数中一并计算该分数；但这里将其保持独立，使它像 `heuristic_score` 一样成为可插拔的评分函数


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch05/CH05_F08_raschka.webp" width=600>

- 上图来自上一章，展示了如何生成下一个词元
- 这里我们只关注不同下一词元候选项的分数，而不实际生成词元


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch05/CH05_F09_raschka.webp" width=600>

- 为了简单起见，上图展示的是 logits 分数；使用概率（词元似然）会更容易解释。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch05/CH05_F10_raschka.webp" width=600>

- 概率分数的问题在于其中大多数都接近零（下一节会解决这个问题）
- 假设某个 LLM 生成答案“Germany's capital is Berlin”，分词后对应 5 个词元
- LLM 会为这些词元分配概率


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch05/CH05_F11_raschka.webp" width=600>

In [ ]:
@torch.inference_mode()
def calc_next_token_probas(model, tokenizer, prompt, device, show=True):

    token_ids = torch.tensor(tokenizer.encode(prompt), device=device)

    # 以类似文本生成函数的方式获取 logits 和概率
    logits = model(token_ids.unsqueeze(0)).squeeze(0)
    all_probas = torch.softmax(logits, dim=-1)

    # 需要评分的位置（这里为全部）
    t_idx = torch.arange(0, token_ids.shape[0] - 1, device=device)

    # 因为已有文本，所以知道真实的下一词元
    next_ids = token_ids[1:]

    # 获取每个下一词元的概率
    next_token_probas = all_probas[t_idx, next_ids]

    # 序列似然是各概率分数的乘积
    prod_next_token_probas = torch.prod(next_token_probas)

    if show:
        print("Next-token probabilities:", next_token_probas)
        print("Joint probability:", prod_next_token_probas)

    else:
        return next_token_probas, prod_next_token_probas

\begin{align}
P(x_1, x_2, \dots, x_T \mid W)
&= \prod_{t=1}^{T} P(x_t \mid x_{1:t-1}, W) \\
&= P(x_1 \mid W) \, \cdot \,
   P(x_2 \mid x_1, W) \, \cdot \,
   P(x_3 \mid x_{1:2}, W) \,
   \cdots \,
   P(x_T \mid x_{1:T-1}, W)
\end{align}

- 其中，$W$ 表示模型参数（学习到的权重），$x$ 表示词元序列，$T$ 表示词元数量
- 由于下面的数值非常接近零，在 CPU、MPS 和 CUDA 设备上会略有差异


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch05/CH05_F12_raschka.webp" width=600>

In [ ]:
torch.set_printoptions(precision=4, sci_mode=True)

In [ ]:
calc_next_token_probas(
        model, tokenizer, device=device,
        prompt="The capital of Germany is Berlin"
    )

In [ ]:
calc_next_token_probas(
        model, tokenizer, device=device,
        prompt="The capital of Germany is Bridge"
    )

In [ ]:
calc_next_token_probas(
        model, tokenizer, device=device,
        prompt="The capital of Germany is Hamburg"
    )

&nbsp;
## 5.5 从词元概率分数到对数概率

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch05/CH05_F13_raschka.webp" width=600>

- 这里再引入一个概念：对数概率（简称 logprob）。它与上一节通过 softmax 函数计算的概率类似，只是对概率取了对数。

In [ ]:
torch.set_printoptions(precision=4, sci_mode=False)

In [ ]:
logits = torch.linspace(-2, 2, steps=7)
probas = torch.softmax(logits, dim=-1)
print(probas)

In [ ]:
print(torch.log(probas))

In [ ]:
log_probas = torch.log_softmax(logits, dim=-1)
print(log_probas)

In [ ]:
plt.figure(figsize=(9, 4))

# logits
plt.subplot(1, 3, 1)
plt.bar(range(len(logits)), logits, color="C0", alpha=0.7)
plt.title("Logits")
plt.xlabel("Token index")
plt.ylabel("Value")
plt.grid(alpha=0.3)

# Softmax
plt.subplot(1, 3, 2)
plt.bar(range(len(probas)), probas, color="C1", alpha=0.7)
plt.title("torch.softmax(logits)")
plt.xlabel("Token index")
plt.ylabel("Probability")
plt.ylim(0, 1)
plt.grid(alpha=0.3)

# 对数 Softmax
plt.subplot(1, 3, 3)
plt.bar(range(len(log_probas)), log_probas, color="C2", alpha=0.7)
plt.title("torch.log_softmax(logits)")
plt.xlabel("Token index")
plt.ylabel("Log-probability")
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("logits_softmax_log_softmax.pdf")
plt.show()

- 对数概率处于更合理的数值范围，有利于进行数值稳定的计算。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch05/CH05_F15_raschka.webp" width=600>

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch05/CH05_F16_raschka.webp" width=600>

\begin{align}
\log P(x_1, x_2, \dots, x_T \mid W)
&= \log \prod_{t=1}^{T} P(x_t \mid x_{1:t-1}, W) \\
&= \sum_{t=1}^{T} \log P(x_t \mid x_{1:t-1}, W) \\
&= \log P(x_1 \mid W)
   + \log P(x_2 \mid x_1, W)
   + \log P(x_3 \mid x_{1:2}, W)
   + \cdots
   + \log P(x_T \mid x_{1:T-1}, W).
\end{align}

In [ ]:
@torch.inference_mode()
def calc_next_token_logprobas(model, tokenizer, prompt, device, show=True):

    token_ids = torch.tensor(tokenizer.encode(prompt), device=device)

    logits = model(token_ids.unsqueeze(0)).squeeze(0)
    # 现在使用 log_softmax
    all_logprobas = torch.log_softmax(logits, dim=-1)

    t_idx = torch.arange(0, token_ids.shape[0] - 1, device=device)
    next_ids = token_ids[1:]
    next_token_logprobas = all_logprobas[t_idx, next_ids]

    # 将乘积替换为求和
    sum_next_token_logprobas = torch.sum(next_token_logprobas)

    if show:
        print("Next-token log-probabilities:", next_token_logprobas)
        print("Joint log-probability:", sum_next_token_logprobas)
    else:
        return next_token_logprobas, sum_next_token_logprobas

In [ ]:
calc_next_token_logprobas(
        model, tokenizer, device=device,
        prompt="The capital of Germany is Berlin"
    )

In [ ]:
calc_next_token_logprobas(
        model, tokenizer, device=device,
        prompt="The capital of Germany is Bridge"
    )

In [ ]:
calc_next_token_logprobas(
        model, tokenizer, device=device,
        prompt="The capital of Germany is Hamburg"
    )

&nbsp;
## 5.6 使用对数概率评估模型置信度

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch05/CH05_F17_raschka.webp" width=600>

- 将对数概率评分转换为可用的评分函数，形式类似启发式评分
- 与上一节相比做两项调整：（1）只对答案评分，不对提示词评分；（2）对各词元分数取平均值


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch05/CH05_F18_raschka.webp" width=600>

In [ ]:
example_prompt = "What is the capital of Germany?"
example_answer = " The capital of Germany is Berlin."

next_token_logprobas, sum_next_token_logprobas = calc_next_token_logprobas(
        model, tokenizer, device=device,
        prompt=example_prompt+example_answer,
        show=False
    )

print("Next-token logprobas:", next_token_logprobas)
print("Joint log-probability:", sum_next_token_logprobas)

In [ ]:
print(len(tokenizer.encode(example_answer)))

In [ ]:
last_7 = next_token_logprobas[-7:]
print(last_7)

In [ ]:
print(torch.mean(last_7))

In [ ]:
@torch.inference_mode()
def avg_logprob_answer(model, tokenizer, prompt, answer, device="cpu"):

    # 分别编码提示词和答案词元，以便稍后获取提示词长度
    prompt_ids = tokenizer.encode(prompt)
    answer_ids = tokenizer.encode(answer)
    full_ids = torch.tensor(prompt_ids + answer_ids, device=device)

    # 与之前的 calc_next_token_logprobas 相同
    logits = model(full_ids.unsqueeze(0)).squeeze(0)
    logprobs = torch.log_softmax(logits, dim=-1)

    # 与答案词元对应的位置索引范围
    start = len(prompt_ids) - 1
    end = full_ids.shape[0] - 1

    # 与之前相同，只是改用 start 和 end
    t_idx = torch.arange(start, end, device=device)
    next_tokens = full_ids[start + 1 : end + 1]
    next_token_logps = logprobs[t_idx, next_tokens]

    # 对答案词元分数取平均
    return torch.mean(next_token_logps)

In [ ]:
score_1 = avg_logprob_answer(
    model, tokenizer,
    prompt="What is the capital of Germany?",
    answer=" The capital of Germany is Berlin.",
    device=device
)
print(score_1)

In [ ]:
score_2 = avg_logprob_answer(
    model, tokenizer,
    prompt="What is the capital of Germany?",
    answer=" The capital of Germany is Bridge.",
    device=device
)
print(score_2)

In [ ]:
avg_logprob_answer(
    model, tokenizer,
    prompt=prompt_cot,
    answer=response_1,
    device=device
)

In [ ]:
avg_logprob_answer(
    model, tokenizer,
    prompt=prompt_cot,
    answer=response_2,
    device=device
)

&nbsp;
## 5.7 通过迭代反馈进行自我改进

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch05/CH05_F19_raschka.webp" width=600>

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch05/CH05_F20_raschka.webp" width=600>

In [ ]:
raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)

torch.manual_seed(123)
initial_response = generate_text_stream_concat_flex(
    model, tokenizer, prompt, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_top_p_stream_cache,
    temperature=0.7,
    top_p=0.9, 
)

In [ ]:
def make_critique_prompt(raw_prompt, draft):
    return (
        "You are a meticulous reviewer. Identify logical errors, missing "
        "steps, or arithmetic mistakes. If the answer seems correct, "
        "say so briefly. Then propose a concise plan to fix issues.\n\n"
        f"Question:\n{raw_prompt}\n\n"
        f"Draft answer:\n{draft}\n\n"
        "Write a short critique and bullet-point fix plan "
        "(under ~120 words).\n"
        "Critique:"
    )


critique_prompt = make_critique_prompt(raw_prompt, initial_response)
torch.manual_seed(123)
critique = generate_text_stream_concat_flex(
    model, tokenizer, critique_prompt, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_top_p_stream_cache,
    temperature=0.7,
    top_p=0.9, 
)

In [ ]:
def make_refine_prompt(raw_prompt, draft, critique):
    return (
        "Revise the answer using the critique. Keep it concise and "
        "end with a final boxed result: \\boxed{ANSWER}\n\n"
        f"Question:\n{raw_prompt}\n\n"
        f"Previous answer:\n{draft}\n\n"
        f"Critique:\n{critique}\n\n"
        "Revised answer:"
    )

refine_prompt = make_refine_prompt(raw_prompt, initial_response, critique)
torch.manual_seed(123)
revised_answer = generate_text_stream_concat_flex(
    model, tokenizer, refine_prompt, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_top_p_stream_cache,
    temperature=0.7,
    top_p=0.9, 
)

&nbsp;
## 5.8 编写自我改进循环

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch05/CH05_F21_raschka.webp" width=600>

In [ ]:
def self_refinement_loop(
    model,
    tokenizer,
    raw_prompt,
    device,
    iterations=2,
    max_response_tokens=2048,
    max_critique_tokens=256,
    score_fn=None,
    prompt_renderer=render_prompt,
    prompt_suffix="",
    verbose=False,
    temperature=0.7,
    top_p=0.9,
):
    steps = []

    # 初始回复（草稿）
    prompt = prompt_renderer(raw_prompt) + prompt_suffix
    current_full = generate_text_stream_concat_flex(
        model=model,
        tokenizer=tokenizer,
        prompt=prompt,
        device=device,
        max_new_tokens=max_response_tokens,
        verbose=False,
        generate_func=generate_text_top_p_stream_cache,
        temperature=temperature,
        top_p=top_p,
    )

    current_extracted = extract_final_candidate(
        current_full, fallback="number_then_full"
    )
    if score_fn:
        current_score = score_fn(answer=current_full, prompt=prompt)
    else:
        current_score = 0.0

    # 运行一次或多次迭代
    for it in range(iterations):
        draft_before_full = current_full
        draft_before_extracted = current_extracted
        score_before = current_score

        # 批评回复
        critique_prompt = make_critique_prompt(
            raw_prompt, draft_before_full
        )
        critique_full = generate_text_stream_concat_flex(
            model=model,
            tokenizer=tokenizer,
            prompt=critique_prompt,
            device=device,
            max_new_tokens=max_critique_tokens,
            verbose=False,
            generate_func=generate_text_top_p_stream_cache,
            temperature=temperature,
            top_p=top_p,
        )

        # 改进回复
        refine_prompt = make_refine_prompt(
            raw_prompt, draft_before_full, critique_full
        )
        revised_full = generate_text_stream_concat_flex(
            model=model,
            tokenizer=tokenizer,
            prompt=refine_prompt,
            device=device,
            max_new_tokens=max_response_tokens,
            verbose=False,
            generate_func=generate_text_top_p_stream_cache,
            temperature=temperature,
            top_p=top_p,
        )

        revised_extracted = extract_final_candidate(
            revised_full, fallback="number_then_full"
        )
        if score_fn:
            revised_score = score_fn(
                answer=revised_full, prompt=prompt  # 这里仍使用原始提示词
            )
        else:
            revised_score = 0.0
        
        # 记录结果
        step = {
            "iteration": it + 1,
            "draft_full": draft_before_full,
            "draft_extracted": draft_before_extracted,
            "critique": critique_full,
            "revised_full": revised_full,
            "revised_extracted": revised_extracted,
            "score_before": score_before,
            "score_after": revised_score,
        }
        steps.append(step)

        if verbose:
            print(
                f"[Refinement {it+1}/{iterations}]"
                f"\nCurrent: {draft_before_extracted}"
                f"\nRevised: {revised_extracted}"
                f"\nScore before: {score_before:.3f}"
                f"\nScore after: {revised_score:.3f}"
                f"\n{'=' * 25}"
            )

        # 如果修改后的回复不更差，则接受它
        if revised_score >= current_score:
            current_full = revised_full
            current_extracted = revised_extracted
            current_score = revised_score

    return {
        "final_full": current_full,
        "final_extracted": current_extracted,
        "steps": steps,
    }

In [ ]:
from functools import partial

avg_logprob_score = partial(
    avg_logprob_answer,
    model=model,
    tokenizer=tokenizer,
    device=device
)

In [ ]:
torch.manual_seed(1)

results_logprob = self_refinement_loop(
    model=model,
    tokenizer=tokenizer,
    raw_prompt=raw_prompt,
    device=device,
    iterations=2,
    max_response_tokens=2048,
    max_critique_tokens=256,
    score_fn=avg_logprob_score,
    verbose=True,
    temperature=0.7,
    top_p=0.9,
)

In [ ]:
print(results_logprob["final_extracted"])

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch05/CH05_F22_raschka.webp" width=600>

&nbsp;
## 5.9 小结

- 本节没有代码